# 🇸🇩 استوديو استنساخ وتوليد الصوت السوداني الذكي (XTTS-v2)
### مُهندس الصوت السوداني الأصيل — استنساخ النبرة والمخارج التلقائية من مكتبة التسجيلات المرجعية
---
**طريقة الاستخدام السريعة (3 خطوات فقط):**
1. اضغط زر التشغيل `▶` على **الخلية 1** (تثبيت المحرك والربط).
2. اضغط زر التشغيل `▶` على **الخلية 2** (تحميل النموذج العصبي XTTS-v2 على T4 GPU).
3. اضغط زر التشغيل `▶` على **الخلية 3** (الواجهة التفاعلية لتوليد وسماع الصوت فوراً).

## 1️⃣ الخلية الأولى: تثبيت محرك الصوت الحديث وربط Google Drive
*(تستغرق حوالي دقيقة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [ ]:
import os
# تفعيل الموافقة التلقائية على ترخيص النموذج العصبي
os.environ['COQUI_TOS_AGREED'] = '1'

# تثبيت حزمة coqui-tts الحديثة المتوافقة 100% مع بايثون الجديد في Colab
!pip install -q coqui-tts soundfile gradio

# ربط Google Drive للوصول للعينات المرجعية والمعجم السوداني
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تثبيت محرك الصوت coqui-tts وربط Google Drive بنجاح!')

## 2️⃣ الخلية الثانية: تحميل نموذج XTTS-v2 العصبي على كرت الشاشة (GPU)
*(تستغرق دقيقة واحدة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

import torch
from TTS.api import TTS
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 المحرك يعمل على: {device}')
if device == 'cuda':
    print(f'⚡ كرت الشاشة المكتشف: {torch.cuda.get_device_name(0)}')

print('⏳ جاري تحميل نموذج Coqui XTTS-v2 العصبي...')
xtts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

# مسار العينات المرجعية والمعجم السوداني المستخرج من الكتب
REF_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references'
KB_PATH = '/content/drive/MyDrive/Sudanese_Voice_Project/sudanese_master_knowledge_base.json'

sudanese_kb = {}
if os.path.exists(KB_PATH):
    with open(KB_PATH, 'r', encoding='utf-8') as f:
        sudanese_kb = json.load(f)
    print(f'📚 تم تحميل المعجم اللغوي السوداني ({sudanese_kb.get("metadata", {}).get("total_extracted_words", 0):,} كلمة)')

print('\n✅ النموذج جاهز للعمل والإنتاج الصوتي الفوري!')

## 3️⃣ الخلية الثالثة: الواجهة التفاعلية الفورية (Gradio Web UI) والتجربة المباشرة
*(شغل هذه الخلية وستفتح لك واجهة كاملة بالمتصفح لكتابة وسماع أي نص سوداني فوراً)*

In [ ]:
import gradio as gr
import os
from IPython.display import Audio, display

# قاموس الأصوات المرجعية السودانية الأصيلة
PRESETS = {
    '🎙️ بودكاست وإعلام رصين (بودكاست السودان)': os.path.join(REF_DIR, 'podcast_alsudanpodcast.wav'),
    '🎭 دراما ومحادثة يومية حية': os.path.join(REF_DIR, 'daily_dialogue_drama.wav'),
    '😄 فكاهة وشخصيات طريفة (شيلابان)': os.path.join(REF_DIR, 'comedy_shailaban.wav'),
    '☕ وناسة وجلسة شبابية خفيفة': os.path.join(REF_DIR, 'wanasa_abubakr.wav'),
    '📖 سرد قصصي هادئ ومتزن': os.path.join(REF_DIR, 'ref_breaklogic_calm.wav')
}

def calibrate_text(text):
    """تطبيق قواعد المعجم السوداني (قلب القاف گ والتسكين)"""
    if 'phonetic_rules' in sudanese_kb and 'qaf_to_gaf_mapping' in sudanese_kb['phonetic_rules']:
        for orig, gaf_w in sudanese_kb['phonetic_rules']['qaf_to_gaf_mapping'].items():
            text = text.replace(orig, gaf_w)
    return text

def generate_sudanese_voice(text, speaker_choice, custom_audio):
    ref_wav = custom_audio if (custom_audio and os.path.exists(custom_audio)) else PRESETS.get(speaker_choice, list(PRESETS.values())[0])
    
    calibrated = calibrate_text(text)
    out_path = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'
    
    print(f'🎙️ توليد بصوت: {os.path.basename(ref_wav)}')
    print(f'📝 النص: {calibrated}')

    xtts.tts_to_file(
        text=calibrated,
        speaker_wav=ref_wav,
        language='ar',
        file_path=out_path
    )
    return out_path

# إطلاق واجهة Gradio التفاعلية برابط عام ومحلي
demo = gr.Interface(
    fn=generate_sudanese_voice,
    inputs=[
        gr.Textbox(label='✍️ اكتب أي نص أو حوار باللهجة السودانية:', value='أهلاً بيكم يا جماعة، دحين الموضوع دا سَمِحْ بالحيل، وقاعدين هنا عشان نتونس فيهو برواقة ونفهم الحاصل شنو.', lines=3),
        gr.Dropdown(choices=list(PRESETS.keys()), value=list(PRESETS.keys())[0], label='🗣️ اختر المتحدث السوداني المرجعي:'),
        gr.Audio(sources=['upload'], type='filepath', label='🎙️ أو ارفع تسجيلك الصوتي الخاص لاستنساخ حنجرتك بدقة (اختياري):')
    ],
    outputs=gr.Audio(label='🔊 الاستماع للمقطع السوداني المستنسخ وتحميله:', type='filepath'),
    title='🇸🇩 استوديو الصوت والذكاء الاصطناعي السوداني',
    description='توليد نطق سوداني واقعي 100% باستخدام الذكاء الاصطناعي العصبي Coqui XTTS-v2'
)

demo.launch(share=True, debug=True)
